In [ ]:
import duckdb
from pathlib import Path
import os

def extract_behavioral_features(user_input_path, dev_mode=False):
    """
    Extracts time-bucketed behavioral features from logon logs.
    
    Parameters:
    - user_input_path: The filename (in PROD) or the full local path (in DEV).
    - dev_mode: Boolean toggle. If True, bypasses the strict security sandbox.
    """
    
    # =====================================================================
    # 1. THE SECURITY SANDBOX & DEV OVERRIDE
    # =====================================================================
    if dev_mode:
        print("⚠️ WARNING: Running in DEV_MODE. Security sandbox disabled.")
        # In Dev Mode, we trust the local path the developer provided
        target_file = Path(user_input_path).resolve()
        
        # We still want to catch basic typos locally!
        if not target_file.exists():
            raise FileNotFoundError(f"DEV ERROR: Cannot find file at {target_file}")
            
    else:
        # PROD MODE: The Strict Sandbox
        safe_directory = Path("/path/to/your/secure/datasets/").resolve()
        
        # In PROD, user_input_path should ONLY be a filename, not a full path
        target_file = (safe_directory / user_input_path).resolve()
        
        # Validation 1: Directory Traversal Check
        if not target_file.is_relative_to(safe_directory):
            raise ValueError("SECURITY ALERT: Invalid file path detected. Access Denied.")
            
    # Validation 2: File Type Check (Enforced in BOTH Dev and Prod)
    if target_file.suffix != '.csv':
        raise ValueError("SECURITY ALERT: Only .csv files are allowed.")

    # =====================================================================
    # 2. THE SQL ARCHITECTURE (Timezones, Bucketing, and Math)
    # =====================================================================
    query = f"""
    -- STEP 1: The CTE (Time Bucketing / Sessionization)
    WITH SessionizedLogs AS (
        SELECT 
            user,
            TIME_BUCKET(
                INTERVAL '5 minutes', 
                TRY_CAST(date AS TIMESTAMP)
            ) AS session_time
            
        FROM read_csv_auto('{target_file}')
        WHERE activity = 'Logon' 
          AND user != 'SYSTEM'
          
        GROUP BY user, session_time
    )
    
    -- STEP 2: The Feature Matrix Extraction
    SELECT 
        user, 
        COUNT(*) AS total_logon_sessions,
        
        SUM(
            CASE 
                WHEN DAYOFWEEK(session_time) IN (0, 6) THEN 1
                WHEN EXTRACT(hour FROM session_time) < 6 OR EXTRACT(hour FROM session_time) >= 18 THEN 1 
                ELSE 0 
            END
        ) AS off_hours_sessions,
        
        SUM(
            CASE 
                WHEN DAYOFWEEK(session_time) IN (0, 6) THEN 1
                WHEN EXTRACT(hour FROM session_time) < 6 OR EXTRACT(hour FROM session_time) >= 18 THEN 1 
                ELSE 0 
            END
        ) * 1.0 / NULLIF(COUNT(*), 0) AS off_hours_ratio

    FROM SessionizedLogs
    GROUP BY user;
    """
    
    # =====================================================================
    # 3. EXECUTION
    # =====================================================================
    
    return duckdb.sql(query).df()




In [ ]:
df = extract_behavioral_features('CERT_dataset/r4.2/logon.csv', dev_mode=True)

In [ ]:
df.head()